# Interactive Web Application: Dual-Engine Fabric Thread Density Analyzer

**The mechanistic justification for this step is:** Construct an interactive, production-ready web application interface and embedded server pipeline that enables users to upload custom fabric imagery, configure spatial calibration, run polar Radon skew rectification, and visualize 2D spectral and 1D projection diagnostics in real-time.

In [1]:
# The mechanistic justification for this step is: configure global environment paths, hardware device utilization, and initialize output directory routing.
import os
import sys
import json
import time
import urllib.request
import threading
import numpy as np
from PIL import Image

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"

INPUT_DIR = "/kaggle/input/fabric-data" if os.path.exists("/kaggle/input/fabric-data") else "data"
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "results"
os.makedirs(WORKING_DIR, exist_ok=True)

print("Device:", DEVICE)
if torch is not None and DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated Memory (MB):", f"{torch.cuda.memory_allocated() / (1024**2):.2f}")
print("Input Directory:", INPUT_DIR)
print("Working Directory:", WORKING_DIR)

Device: cpu
Input Directory: data
Working Directory: results


## REST API Architecture & Zero-External-Dependency Engine

**The mechanistic justification for this step is:** Decouple user interaction from python command execution by exposing a RESTful HTTP protocol served by Python standard library `http.server` and `socketserver.ThreadingMixIn`. This eliminates external web framework dependencies (e.g. Flask/FastAPI/Uvicorn) while providing concurrent analysis for multiple users.

In [2]:
# The mechanistic justification for this step is: verify the web application server components and modules exist within the workspace.
server_file = os.path.join("web_app", "server.py")
engine_file = os.path.join("web_app", "engine.py")
index_file = os.path.join("web_app", "index.html")

print("server.py exists:", os.path.exists(server_file))
print("engine.py exists:", os.path.exists(engine_file))
print("index.html exists:", os.path.exists(index_file))
print("server.py size (bytes):", os.path.getsize(server_file))
print("engine.py size (bytes):", os.path.getsize(engine_file))
print("index.html size (bytes):", os.path.getsize(index_file))

server.py exists: True
engine.py exists: True
index.html exists: True
server.py size (bytes): 10831
engine.py size (bytes): 19341
index.html size (bytes): 44192


## Embedded Server Execution & Lifecycle Management

**The mechanistic justification for this step is:** Spawn the threaded HTTP server in a background daemon thread within the notebook kernel, enabling programmatic API validation and interactive inline previewing without blocking the notebook workflow.

In [3]:
# The mechanistic justification for this step is: initialize and bind the ThreadedHTTPServer on an available local port with clean environment proxy routing.
os.environ.pop("http_proxy", None)
os.environ.pop("HTTP_PROXY", None)

from web_app.server import ThreadedHTTPServer, FabricAnalysisHandler

PORT = 8085
HOST = "127.0.0.1"

server = ThreadedHTTPServer((HOST, PORT), FabricAnalysisHandler)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
time.sleep(0.5)

print("Server running:", server_thread.is_alive())
print("Bound address:", f"http://{HOST}:{PORT}/")

Server running: True
Bound address: http://127.0.0.1:8085/


## Programmatic REST API Verification Suite

**The mechanistic justification for this step is:** Empirically validate every REST endpoint against synthetic and real physical fabric benchmarks, measuring HTTP status codes, latency in milliseconds, thread density accuracy, and payload integrity.

In [4]:
# The mechanistic justification for this step is: execute end-to-end integration tests covering health, presets, synthetic tilted analysis, microscopic analysis, and PDF report downloads.
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
BASE_URL = f"http://{HOST}:{PORT}"

# 1. Health Endpoint
with opener.open(f"{BASE_URL}/api/health") as resp:
    health_res = json.loads(resp.read().decode("utf-8"))
    print("Health Status:", resp.status, "Service:", health_res.get("service"))

# 2. Presets Endpoint
with opener.open(f"{BASE_URL}/api/presets") as resp:
    presets_res = json.loads(resp.read().decode("utf-8"))
    presets = presets_res.get("presets", [])
    print("Presets Count:", len(presets))
    for p in presets:
        print(f"  Preset: {p['id']} | Ground Truth: {p['ground_truth']} | Scale: {p['pixels_per_cm']} px/cm")

# 3. Analyze Preset: Synthetic Skewed +12°
payload_skew = {
    "preset_id": "synthetic_skewed_12",
    "pixels_per_cm": 400.0,
    "auto_skew": True,
    "prominence": 0.012,
    "weave_hint": "plain"
}
t0 = time.perf_counter()
req = urllib.request.Request(
    f"{BASE_URL}/api/analyze",
    data=json.dumps(payload_skew).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)
with opener.open(req) as resp:
    skew_res = json.loads(resp.read().decode("utf-8"))
    lat_skew = (time.perf_counter() - t0) * 1000.0
    r = skew_res["results"]
    print(f"Synthetic (+12° Skew) -> Warp: {r['warp_ends_per_cm']} ends/cm, Weft: {r['weft_picks_per_cm']} picks/cm, Skew: {r['detected_skew_deg']}°, Quality: {r['confidence_score']}%, Latency: {lat_skew:.2f} ms")

# 4. Analyze Preset: FabricNET W012 (Plain Weave)
payload_w012 = {
    "preset_id": "fabricnet_w012",
    "pixels_per_cm": 1280.0,
    "auto_skew": False,
    "prominence": 0.010,
    "weave_hint": "plain"
}
t0 = time.perf_counter()
req = urllib.request.Request(
    f"{BASE_URL}/api/analyze",
    data=json.dumps(payload_w012).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)
with opener.open(req) as resp:
    w012_res = json.loads(resp.read().decode("utf-8"))
    lat_w012 = (time.perf_counter() - t0) * 1000.0
    r = w012_res["results"]
    print(f"FabricNET W012 (Plain) -> Warp: {r['warp_ends_per_cm']} ends/cm, Weft: {r['weft_picks_per_cm']} picks/cm, Quality: {r['confidence_score']}%, Latency: {lat_w012:.2f} ms")

# 5. Report Downloads
with opener.open(f"{BASE_URL}/reports/warp_weft_report_en.pdf") as resp:
    en_pdf_size = len(resp.read())
with opener.open(f"{BASE_URL}/reports/warp_weft_report_ar.pdf") as resp:
    ar_pdf_size = len(resp.read())

print("English PDF Size (bytes):", en_pdf_size)
print("Arabic PDF Size (bytes):", ar_pdf_size)

Health Status: 200 Service: Warp-Weft Thread Density Analyzer
Presets Count: 4
  Preset: synthetic_skewed_12 | Ground Truth: 36 Warp / 30 Weft (Tilt: +12°) | Scale: 400.0 px/cm
  Preset: fabricnet_w012 | Ground Truth: 38 Warp / 33 Weft (Exact Match) | Scale: 1280.0 px/cm
  Preset: fabricnet_w001 | Ground Truth: 32 Warp / 30 Weft (Float Ridges) | Scale: 1280.0 px/cm
  Preset: synthetic_skewed_15 | Ground Truth: 32 Warp / 28 Weft (Tilt: +15°) | Scale: 400.0 px/cm
Synthetic (+12° Skew) -> Warp: 36 ends/cm, Weft: 30 picks/cm, Skew: 11.5°, Quality: 69.5%, Latency: 243.64 ms
FabricNET W012 (Plain) -> Warp: 40 ends/cm, Weft: 30 picks/cm, Quality: 37.5%, Latency: 193.83 ms
English PDF Size (bytes): 20118458
Arabic PDF Size (bytes): 1864848


## Interactive Interface Display & Standalone Execution

**The mechanistic justification for this step is:** Provide direct embedded UI interaction inside Jupyter via `IPython.display.IFrame` and document standalone terminal execution commands for external workstation access.

In [5]:
# The mechanistic justification for this step is: render the live application interface inside an IPython IFrame widget for inline testing.
try:
    from IPython.display import IFrame, display
    iframe = IFrame(src=f"http://127.0.0.1:{PORT}/", width="100%", height="850")
    display(iframe)
except ImportError:
    pass

### Standalone Execution Instructions

To run the web application independently outside Jupyter on any workstation:

```bash
# Start web server on port 8080 (or port 8081 fallback)
python3 web_app/server.py
```

Then open any modern web browser and navigate to:
```
http://localhost:8080/
```